In [4]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [5]:
model_id = "google/gemma-4-E2B"
adapter_path = "./gemma4-darija-qlora"
compute_dtype = torch.float16

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

In [6]:
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map={"":0},
    dtype=compute_dtype,
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)
model.eval()


OSError: The paging file is too small for this operation to complete. (os error 1455)

In [ ]:
prompt = "واش راك اليوم؟"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))